# Section F0 v2 — PyTorch semantics diagnosticThree parts, about 45 minutes total.- **Part 1 (12 items) — prediction.** You are given a snippet. Predict the exact output, shape,  or error *before* running it. This measures recognition: can you read unfamiliar torch and  know what it does.- **Part 2 (5 items) — generation.** Blank editor, no docs, no web. Implement the function from  primitives. Each one is differential-tested against the real PyTorch builtin over 200  randomized shapes, so the answer key is `torch`, not me. This measures retrieval.- **Part 3 (3 items) — assertion writing.** Given a silent bug, write the assertion that would  have caught it before you ran anything.**Rules.** No documentation, no search, no autocomplete suggestions accepted for Parts 2 and 3.Write the prediction down physically (a scratch cell is fine) before running. A prediction youdid not commit to does not count as correct.Verified against torch 2.14. Behaviour of every item was confirmed by execution.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import base64, json

print("torch", torch.__version__)

def show(label, fn):
    "Run one expression, printing either its value or the exception it raises."
    try:
        print(f"{label}\n   -> {fn()}\n")
    except Exception as e:
        print(f"{label}\n   -> {type(e).__name__}: {str(e)[:110]}\n")

_R = {}   # populated below

def reveal(n):
    print(base64.b64decode(_R[n]).decode())

In [ ]:
_R = {
    1: "UDEuIFN0cmlkZXM6ICgxMiw0LDEpIGNvbnRpZ3VvdXMsICg0LDEyLDEpIGFmdGVyIHRyYW5zcG9zZS4KICAgIC52aWV3KDYsNCkgICAgLT4gUnVudGltZUVycm9yLgogICAgLnJlc2hhcGUoNiw0KSAtPiB0b3JjaC5TaXplKFs2LCA0XSkuCgpSVUxFLiBNZXJnaW5nIHR3byBheGVzIGlzIGEgZnJlZSBtZXRhZGF0YS1vbmx5IHZpZXcgaWYgYW5kIG9ubHkgaWYKICAgICAgc3RyaWRlKG91dGVyKSA9PSBzdHJpZGUoaW5uZXIpICogc2l6ZShpbm5lcikuCiAgICAgIENvbnRpZ3VvdXM6IHN0cmlkZTAgPSAxMiA9IHN0cmlkZTEgKiBzaXplMSA9IDQgKiAzLiBNZXJnZXMuCiAgICAgIFRyYW5zcG9zZWQ6IHN0cmlkZTAgPSA0LCBidXQgc3RyaWRlMSAqIHNpemUxID0gMTIgKiAyID0gMjQuIERvZXMgbm90IG1lcmdlLgogICAgICByZXNoYXBlKCkgc2lsZW50bHkgZmFsbHMgYmFjayB0byBhIGNvcHkuIFRoYXQgaXMgdGhlIGNvc3QgeW91IGRpZCBub3QgYXNrIGZvcjoKICAgICAgdmlldyBmYWlscyBsb3VkbHksIHJlc2hhcGUgc3VjY2VlZHMgcXVpZXRseSBhbmQgYWxsb2NhdGVzLg==",
    2: "UDIuIChhKSB0b3JjaC5TaXplKFsyLCA2NCwgMTAyNF0pLiAgIChiKSBSdW50aW1lRXJyb3IuICAgKGMpIHRvcmNoLlNpemUoWzIsIDgsIDgxOTJdKS4KCldIWS4gcSBzdHJpZGVzIGFyZSAoNjU1MzYsIDEwMjQsIDEyOCwgMSkuCiAgICAgKGEpIG1lcmdlcyBoZWFkcyBhbmQgaGVhZF9kaW06IHN0cmlkZShoKSA9IDEyOCA9IHN0cmlkZShkKSAqIHNpemUoZCkgPSAxICogMTI4LiBGcmVlIHZpZXcuCiAgICAgKGIpIGFmdGVyIHBlcm11dGUgdG8gW2IsaCxzLGRdIHN0cmlkZXMgYXJlICg2NTUzNiwgMTI4LCAxMDI0LCAxKS4gTWVyZ2luZyBzIGFuZCBkIG5lZWRzCiAgICAgICAgIHN0cmlkZShzKSA9PSBzdHJpZGUoZCkgKiBzaXplKGQpID0gMTI4LCBidXQgc3RyaWRlKHMpID0gMTAyNC4gRmFpbHMuCiAgICAgKGMpIGNvbnRpZ3VvdXMoKSBtYXRlcmlhbGlzZXMgYSBjb3B5IGZpcnN0LCB0aGVuIHRoZSBtZXJnZSBpcyBsZWdhbC4KClRoaXMgaXMgd2h5IGAuY29udGlndW91cygpYCBhcHBlYXJzIGFmdGVyIGV2ZXJ5IHBlcm11dGUgaW4gYXR0ZW50aW9uIGNvZGUsIGFuZCBpdCBpcyBhIHJlYWwKY29weSBvZiB0aGUgd2hvbGUgYWN0aXZhdGlvbiB0ZW5zb3IsIG5vdCBhIG5vLW9wLg==",
    3: "UDMuIHRvcmNoLlNpemUoWzQsIDRdKS4KClJVTEUuIFNoYXBlcyBhcmUgcmlnaHQtYWxpZ25lZCwgdGhlbiBldmVyeSBzaXplLTEgYXhpcyBpcyBzdHJldGNoZWQuCiAgICAgICg0LDEpIGFuZCAoNCwpIC0+ICg0LDEpIGFuZCAoMSw0KSAtPiAoNCw0KS4KICAgICAgWW91IGFza2VkIGZvciBhIHZlY3RvciBhZGQgYW5kIGdvdCBhbiBvdXRlciBwcm9kdWN0LCB3aXRoIG5vIGVycm9yLgoKVGhpcyBpcyB0aGUgc2luZ2xlIG1vc3QgY29tbW9uIHNpbGVudCBidWcgaW4gdG9yY2guIFRoZSBmaXggaXMgYi51bnNxdWVlemUoLTEpLCBnaXZpbmcgKDQsMSkuClRoZSBoYWJpdCB0aGF0IGNhdGNoZXMgaXQ6IGFzc2VydCBvdXQuc2hhcGUgPT0gZXhwZWN0ZWRfc2hhcGUsIG5vdCAidGhlIG51bWJlcnMgbG9vayBmaW5lIi4=",
    4: "UDQuIG1heCgtMSkgcmV0dXJucyBhIG5hbWVkdHVwbGUgKCdtYXgnKSB3aXRoIC52YWx1ZXMgYW5kIC5pbmRpY2VzLCBub3QgYSB0ZW5zb3IuCiAgICBXaXRob3V0IGtlZXBkaW06IFJ1bnRpbWVFcnJvciwgc2l6ZSA1IHZzIDQgYXQgZGltIDEuCiAgICBXaXRoIGtlZXBkaW06ICAgIHRvcmNoLlNpemUoWzQsIDVdKS4KClRIRSBSRUFMIExFU1NPTi4gVGhlIGVycm9yIGhlcmUgaXMgYSBsdWNreSBhY2NpZGVudCBvZiA0ICE9IDUuIE9uIGEgU1FVQVJFIHRlbnNvciB0aGUKc2FtZSBjb2RlIHJ1bnMgYW5kIHNpbGVudGx5IHN1YnRyYWN0cyB0aGUgY29sdW1uIG1heCBmcm9tIHRoZSByb3dzLiBUaGF0IGlzIGEgc3RhYmxlLXNvZnRtYXgKYnVnIHRoYXQgcHJvZHVjZXMgZmluaXRlLCBwbGF1c2libGUsIHdyb25nIG51bWJlcnMuIEFsd2F5cyBrZWVwZGltPVRydWUgb24gdGhlIG1heCB5b3UgYXJlCmFib3V0IHRvIGJyb2FkY2FzdCBhZ2FpbnN0Lg==",
    5: "UDUuIHlbaWR4XSAgICAgICAgIC0+IHNoYXBlICgyLDQpLCByb3dzIDAgYW5kIDIuCiAgICB5W2lkeCwgaWR4XSAgICAtPiBzaGFwZSAoMiwpLCAgdGVuc29yKFswLCAxMF0pLiAgVGhhdCBpcyB5WzAsMF0gYW5kIHlbMiwyXS4KICAgIHlbaWR4XVs6LCBpZHhdIC0+IHNoYXBlICgyLDIpLCB0ZW5zb3IoW1swLDJdLFs4LDEwXV0pLgoKUlVMRS4gTXVsdGlwbGUgaW5kZXggdGVuc29ycyBhcmUgYnJvYWRjYXN0IGFnYWluc3QgZWFjaCBvdGhlciBhbmQgWklQUEVEIGVsZW1lbnR3aXNlLgogICAgICBUaGV5IGRvIG5vdCBmb3JtIGEgY3Jvc3MgcHJvZHVjdC4gVG8gZ2V0IHRoZSBjcm9zcyBwcm9kdWN0IGluIG9uZSBzdGVwLCBtYWtlIHRoZW0KICAgICAgYnJvYWRjYXN0LWluY29tcGF0aWJsZSBvbiBwdXJwb3NlOiB5W2lkeC51bnNxdWVlemUoMSksIGlkeF0gLT4gKDIsMikuCgpUaGlzIGlzIHRoZSBtZWNoYW5pc20gYmVoaW5kIGdhdGhlci1zdHlsZSBsb29rdXBzLCBhbmQgZ2V0dGluZyBpdCBiYWNrd2FyZHMgaXMgaG93IHlvdSBlbmQgdXAKc2VsZWN0aW5nIHRoZSBkaWFnb25hbCBvZiBzb21ldGhpbmcgeW91IG1lYW50IHRvIHNlbGVjdCBhIHN1Ym1hdHJpeCBvZi4=",
    6: "UDYuIHNsaWNpbmcgLT4gdGVuc29yKFtbMS4sMS4sMS5dLFswLiwwLiwwLl1dKSAgIChtIFdBUyBtb2RpZmllZCkKICAgIGJvb2wgbWFzayAtPiB0ZW5zb3IoW1swLiwwLiwwLl0sWzAuLDAuLDAuXV0pIChtIHdhcyBOT1QgbW9kaWZpZWQpCgpSVUxFLiBCYXNpYyBpbmRleGluZyAoaW50ZWdlcnMsIHNsaWNlcywgZWxsaXBzaXMsIE5vbmUpIHJldHVybnMgYSBWSUVXIHNoYXJpbmcgc3RvcmFnZS4KICAgICAgQWR2YW5jZWQgaW5kZXhpbmcgKGluZGV4IHRlbnNvcnMsIGJvb2xlYW4gbWFza3MpIHJldHVybnMgYSBDT1BZLgoKQm90aCBmYWlsdXJlIGRpcmVjdGlvbnMgYXJlIHNpbGVudC4gV3JpdGluZyB0aHJvdWdoIGEgdmlldyB5b3UgdGhvdWdodCB3YXMgYSBjb3B5IGNvcnJ1cHRzCnVwc3RyZWFtIHN0YXRlOyB3cml0aW5nIHRvIGEgY29weSB5b3UgdGhvdWdodCB3YXMgYSB2aWV3IHNpbGVudGx5IGRpc2NhcmRzIHlvdXIgdXBkYXRlLiBUaGUKc2Vjb25kIGlzIHRoZSBjbGFzc2ljIEtWLWNhY2hlLWRvZXMtbm90LXVwZGF0ZSBidWcu",
    7: "UDcuIHRlbnNvcihbMywgNCwgOV0pLgoKUlVMRS4gb3V0W2ldW2pdID0geFtpXVsgaW5kZXhbaV1bal0gXSBmb3IgZGltPTEuCiAgICAgIFRoZSBpbmRleCB0ZW5zb3IgbXVzdCBoYXZlIHRoZSBzYW1lIG51bWJlciBvZiBkaW1lbnNpb25zIGFzIHgsIGFuZCBtdXN0IG1hdGNoIHggaW4KICAgICAgZXZlcnkgZGltZW5zaW9uIGV4Y2VwdCB0aGUgZ2F0aGVyZWQgb25lLgoKVGhpcyBpcyB0aGUgcHJpbWl0aXZlIGJlaGluZCBjcm9zcy1lbnRyb3B5J3MgTkxMIHRlcm06IGdhdGhlciB0aGUgbG9naXQgYXQgdGhlIGxhYmVsIGluZGV4Cmluc3RlYWQgb2YgYnVpbGRpbmcgYSBvbmUtaG90IGFuZCBkb2luZyBhIG1hdG11bC4=",
    8: "UDguIGluZGV4X2FkZCAgLT4gdGVuc29yKFszLiwgMC4sIDMuXSkgICAoMSArIDIgYWNjdW11bGF0ZWQgYXQgcG9zaXRpb24gMCkKICAgIGluZGV4X3B1dF8gLT4gdGVuc29yKFsyLiwgMC4sIDMuXSkgICAob25lIHdyaXRlciB3aW5zIGF0IHBvc2l0aW9uIDApCgpSVUxFLiBVbmRlciBkdXBsaWNhdGUgaW5kaWNlcywgdGhlIGFjY3VtdWxhdGUgZmFtaWx5IChpbmRleF9hZGRfLCBzY2F0dGVyX2FkZF8sIGJpbmNvdW50KQogICAgICBzdW1zOyBwbGFpbiBhc3NpZ25tZW50IChpbmRleF9wdXRfLCB4W2lkeF0gPSB2KSBrZWVwcyBleGFjdGx5IG9uZSB2YWx1ZSwgYW5kIHdoaWNoIG9uZQogICAgICBpcyBub3QgZ3VhcmFudGVlZCBpbiBnZW5lcmFsIGFuZCBpcyBub25kZXRlcm1pbmlzdGljIG9uIEdQVS4KCldoZXJlIHRoaXMgYml0ZXM6IE1vRSBkaXNwYXRjaCBhbmQgY29tYmluZSwgZW1iZWRkaW5nLWJhZywgYW55IHNjYXR0ZXItYmFzZWQgYWdncmVnYXRpb24uCkNob29zaW5nIHRoZSB3cm9uZyBvbmUgcHJvZHVjZXMgYSBwbGF1c2libGUgdGVuc29yLCBuZXZlciBhbiBlcnJvci4=",
    9: "UDkuIChhKSB0b3JjaC5mbG9hdDMyLiAgKGIpIFJ1bnRpbWVFcnJvciwgbWVhbiBuZWVkcyBhIGZsb2F0IGlucHV0LiAgKGMpIFJ1bnRpbWVFcnJvci4KClJVTEUuIE91dC1vZi1wbGFjZSBvcHMgUFJPTU9URSB0byB0aGUgd2lkZXIgZHR5cGUgYW5kIGdpdmUgeW91IGEgbmV3IHRlbnNvci4KICAgICAgSW4tcGxhY2Ugb3BzIGNhbm5vdCBjaGFuZ2UgdGhlIG91dHB1dCBkdHlwZSwgc28gYW4gaW50IHRlbnNvciBkaXZpZGVkIGluIHBsYWNlIGZhaWxzLgogICAgICBUcnVlIGRpdmlzaW9uIGFsd2F5cyBwcm9tb3RlcyB0byBmbG9hdCBldmVuIGZvciBpbnQgaW5wdXRzOyB1c2UgLy8gZm9yIGZsb29yIGRpdmlzaW9uLgoKQ29yb2xsYXJ5IHdvcnRoIGNhcnJ5aW5nOiBhIG5ldyB0ZW5zb3IgYnVpbHQgd2l0aCB0b3JjaC56ZXJvcyguLi4pIGRvZXMgTk9UIGluaGVyaXQgZHR5cGUgb3IKZGV2aWNlIGZyb20gYW55dGhpbmcuIHRvcmNoLnplcm9zX2xpa2UoeCkgZG9lcy4gTWl4aW5nIHRoZSB0d28gaXMgdGhlIHN0YW5kYXJkIHNvdXJjZSBvZgoiZXhwZWN0ZWQgYWxsIHRlbnNvcnMgb24gdGhlIHNhbWUgZGV2aWNlIiB0aHJlZSBmdW5jdGlvbnMgbGF0ZXIu",
    10: "UDEwLiBUaGUgZmlyc3QgdGhyZWUgYWxsIGdpdmUgdG9yY2guU2l6ZShbMiwgOCwgMTYsIDE2XSkuCiAgICAgUSBAIEsuVCByYWlzZXMsIGFuZCBhbHNvIGVtaXRzIGEgZGVwcmVjYXRpb24gd2FybmluZy4KClJVTEUuIEluIEAgLyBtYXRtdWwsIGFsbCBsZWFkaW5nIGRpbXMgYXJlIEJBVENIIGRpbXMgYW5kIG9ubHkgdGhlIGxhc3QgdHdvIHBhcnRpY2lwYXRlLgogICAgICAuVCBvbiBhIHRlbnNvciB3aXRoIG1vcmUgdGhhbiAyIGRpbXMgcmV2ZXJzZXMgRVZFUlkgZGltZW5zaW9uLCB3aGljaCBpcyBhbG1vc3QgbmV2ZXIKICAgICAgd2hhdCB5b3Ugd2FudDsgaXQgaXMgZGVwcmVjYXRlZCBhbmQgd2lsbCBiZWNvbWUgYW4gZXJyb3IuIFVzZSAubVQgb3IgdHJhbnNwb3NlKC0xLC0yKS4KICAgICAgZWluc3VtIG5hbWVzIHRoZSBheGVzIGV4cGxpY2l0bHksIHNvIGl0IGNhbm5vdCBiZSB3cm9uZyBzaWxlbnRseS4KCllvdXIgRDIgZGVmZWN0IGxpc3QgY29udGFpbnMgIndyb25nIGVpbnN1bSBheGlzIi4gVGhlIGhhYml0IHRoYXQgcmVtb3ZlcyBpdDogcmVhZCB0aGUgb3V0cHV0CnN1YnNjcmlwdCBmaXJzdCBhbmQgY2hlY2sgdGhhdCAoaSkgdGhlIGNvbnRyYWN0ZWQgbGV0dGVyIGFwcGVhcnMgaW4gYm90aCBpbnB1dHMgYW5kIGluIG5laXRoZXIKb3V0cHV0LCBhbmQgKGlpKSBldmVyeSBvdXRwdXQgbGV0dGVyIGFwcGVhcnMgaW4gYXQgbGVhc3Qgb25lIGlucHV0Lg==",
    11: "UDExLiAoYSkgUnVudGltZUVycm9yOiBhIHZhcmlhYmxlIG5lZWRlZCBmb3IgZ3JhZGllbnQgY29tcHV0YXRpb24gd2FzIG1vZGlmaWVkIGJ5IGFuCiAgICAgICAgIGluLXBsYWNlIG9wZXJhdGlvbiwgb3V0cHV0IDAgb2YgRXhwLgogICAgIChiKSBXb3Jrcy4gei5ncmFkID09IHRlbnNvcihbMC4sIDIuLCAyLl0pLgoKUlVMRS4gSW4tcGxhY2UgaXMgbm90IGJhbm5lZCBieSBhdXRvZ3JhZC4gSXQgZmFpbHMgb25seSB3aGVuIHlvdSBjbG9iYmVyIGEgdGVuc29yIHRoYXQgc29tZQogICAgICBiYWNrd2FyZCBmdW5jdGlvbiBTQVZFRC4gZXhwIHNhdmVzIGl0cyBvd24gT1VUUFVUIChkL2R4IGV4cCh4KSA9IGV4cCh4KSksIHNvIG11dGF0aW5nCiAgICAgIHRoZSBvdXRwdXQgYnJlYWtzIGl0LiBtdWwgc2F2ZXMgb25seSB0aGUgb3RoZXIgb3BlcmFuZCwgc28gaXRzIG91dHB1dCBpcyBleHBlbmRhYmxlLgoKTm90ZSB0aGUgZ3JhZGllbnQgaW4gKGIpOiBwb3NpdGlvbiAwIGlzIDAuMCwgYmVjYXVzZSB0aGUgb3ZlcndyaXRlIHNldmVyZWQgdGhhdCBlbGVtZW50IGZyb20KdGhlIGdyYXBoLiBObyBlcnJvciwgYSB3cm9uZyBncmFkaWVudC4gVGhpcyBpcyB0aGUgaW4tcGxhY2UgZmFpbHVyZSBtb2RlIHRoYXQgYWN0dWFsbHkgY29zdHMKeW91IGEgdHJhaW5pbmcgcnVuLg==",
    12: "UDEyLiBbJ3cnLCAnYnVmJ10uICBUaGUgcGxhaW4gYXR0cmlidXRlIGlzIGFic2VudC4KClJVTEUuIE9ubHkgbm4uUGFyYW1ldGVyIGFuZCByZWdpc3RlcmVkIGJ1ZmZlcnMgYXJlIHRyYWNrZWQuIEEgcGxhaW4gdGVuc29yIGF0dHJpYnV0ZQogICAgICAoaSkgaXMgbm90IG1vdmVkIGJ5IC50byhkZXZpY2UpIG9yIC5jdWRhKCksCiAgICAgIChpaSkgaXMgbm90IHNhdmVkIG9yIHJlc3RvcmVkIGJ5IHN0YXRlX2RpY3QgLyBsb2FkX3N0YXRlX2RpY3QsCiAgICAgIChpaWkpIGlzIG5vdCBjb252ZXJ0ZWQgYnkgLmhhbGYoKSAvIC5iZmxvYXQxNigpLgoKQ29uY3JldGVseTogYSBLViBjYWNoZSwgYSByb3RhcnkgY29zL3NpbiB0YWJsZSwgb3IgYSBjYXVzYWwgbWFzayBzdG9yZWQgYXMgc2VsZi54ID0gdG9yY2gu4oCmIGlzCmEgZGV2aWNlIG1pc21hdGNoIGF0IHRoZSBmaXJzdCAuY3VkYSgpIGFuZCBhIHNpbGVudGx5IG1pc3NpbmcgdGVuc29yIGF0IHRoZSBmaXJzdCBjaGVja3BvaW50CnJlc3RvcmUuIFlvdXIgcHJvZ3Jlc3Mgbm90ZXMgYWxyZWFkeSBsaXN0ICJkdHlwZS9kZXZpY2UiIGFzIGEgRDIgZGVmZWN0OyB0aGlzIGlzIGl0cyBtb2R1bGUtbGV2ZWwKZm9ybSwgYW5kIHJlc3RhcnQgY29ycmVjdG5lc3MgaXMgYSBNb2R1bGUgNCB0b3BpYy4=",
}
print("answer key loaded; use reveal(n) after you have committed a prediction")

---# Part 1 — Prediction (12 items)One or two items per primitive family, so that a miss tells us something specific.

### P1. view after transposeWhat does each line print?**Write your prediction down before you run the next cell.**

In [ ]:
x = torch.randn(2, 3, 4)

show("x.stride(),  x.transpose(0,1).stride()", lambda: (x.stride(), x.transpose(0,1).stride()))
show("x.transpose(0,1).view(6,4).shape",       lambda: x.transpose(0,1).view(6,4).shape)
show("x.transpose(0,1).reshape(6,4).shape",    lambda: x.transpose(0,1).reshape(6,4).shape)

In [ ]:
reveal(1)

### P2. the attention reshapeThis is the exact reshape every attention implementation performs. Which of the three succeed?**Write your prediction down before you run the next cell.**

In [ ]:
q = torch.randn(2, 64, 8, 128)          # [batch, seq, heads, head_dim]

show("q.view(2,64,1024).shape",
     lambda: q.view(2, 64, 1024).shape)
show("q.permute(0,2,1,3).view(2,8,8192).shape",
     lambda: q.permute(0, 2, 1, 3).view(2, 8, 8192).shape)
show("q.permute(0,2,1,3).contiguous().view(2,8,8192).shape",
     lambda: q.permute(0, 2, 1, 3).contiguous().view(2, 8, 8192).shape)

In [ ]:
reveal(2)

### P3. the silent broadcastYou intended an elementwise add. What actually happens?**Write your prediction down before you run the next cell.**

In [ ]:
a = torch.randn(4, 1)
b = torch.randn(4)

show("(a + b).shape", lambda: (a + b).shape)

In [ ]:
reveal(3)

### P4. max and keepdimWhat is the type of kk.max(-1), and what do the two subtractions do?**Write your prediction down before you run the next cell.**

In [ ]:
kk = torch.randn(4, 5)

show("type(kk.max(-1)).__name__", lambda: type(kk.max(-1)).__name__)
show("(kk - kk.max(-1).values).shape", lambda: (kk - kk.max(-1).values).shape)
show("(kk - kk.max(-1, keepdim=True).values).shape", lambda: (kk - kk.max(-1, keepdim=True).values).shape)

In [ ]:
reveal(4)

### P5. index tensors are zipped, not crossedGive the shape and the values of each.**Write your prediction down before you run the next cell.**

In [ ]:
y = torch.arange(12).reshape(3, 4)
idx = torch.tensor([0, 2])

show("y[idx]",        lambda: y[idx])
show("y[idx, idx]",   lambda: y[idx, idx])
show("y[idx][:, idx]", lambda: y[idx][:, idx])

In [ ]:
reveal(5)

### P6. view or copyAfter each block, what is m?**Write your prediction down before you run the next cell.**

In [ ]:
def slice_then_add():
    m = torch.zeros(2, 3)
    r = m[0]
    r += 1
    return m

def mask_then_add():
    m = torch.zeros(2, 3)
    s = m[m > -1]
    s += 1
    return m

show("basic slicing", slice_then_add)
show("boolean mask",  mask_then_add)

In [ ]:
reveal(6)

### P7. gatherWhat comes out?**Write your prediction down before you run the next cell.**

In [ ]:
g = torch.arange(12).reshape(3, 4)
gi = torch.tensor([[3], [0], [1]])

show("g.gather(1, gi).flatten()", lambda: g.gather(1, gi).flatten())

In [ ]:
reveal(7)

### P8. duplicate indicesTwo ways to write values into a buffer at possibly-repeated positions. Do they agree?**Write your prediction down before you run the next cell.**

In [ ]:
show("index_add",  lambda: torch.zeros(3).index_add(0, torch.tensor([0,0,2]), torch.tensor([1.,2.,3.])))
show("index_put_", lambda: torch.zeros(3).index_put_((torch.tensor([0,0,2]),), torch.tensor([1.,2.,3.])))

In [ ]:
reveal(8)

### P9. dtype promotionWhich of these three work, and what do they produce?**Write your prediction down before you run the next cell.**

In [ ]:
show("(torch.arange(5) / 2).dtype", lambda: (torch.arange(5) / 2).dtype)
show("torch.arange(5).mean()",      lambda: torch.arange(5).mean())
show("torch.arange(5).div_(2)",     lambda: torch.arange(5).div_(2))

In [ ]:
reveal(9)

### P10. batched matmul and the transpose trapAll three intend Q K^T over the last two dims. Which are correct?**Write your prediction down before you run the next cell.**

In [ ]:
Q = torch.randn(2, 8, 16, 64)
K = torch.randn(2, 8, 16, 64)

show("einsum bhqd,bhkd->bhqk", lambda: torch.einsum('bhqd,bhkd->bhqk', Q, K).shape)
show("Q @ K.transpose(-1,-2)", lambda: (Q @ K.transpose(-1, -2)).shape)
show("Q @ K.mT",              lambda: (Q @ K.mT).shape)
show("Q @ K.T",               lambda: (Q @ K.T).shape)

In [ ]:
reveal(10)

### P11. in-place and the autograd graphOne of these raises and one does not. Which, and why?**Write your prediction down before you run the next cell.**

In [ ]:
def exp_then_inplace():
    z = torch.randn(3, requires_grad=True)
    w = z.exp()
    w[0] = 0.
    w.sum().backward()
    return z.grad

def mul_then_inplace():
    z = torch.randn(3, requires_grad=True)
    w = (z * 2).clone()
    w[0] = 5.
    w.sum().backward()
    return z.grad

show("exp then inplace", exp_then_inplace)
show("mul then inplace", mul_then_inplace)

In [ ]:
reveal(11)

### P12. what is in the state dictWhich of the three attributes appear in state_dict()?**Write your prediction down before you run the next cell.**

In [ ]:
class M(nn.Module):
    def __init__(self):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(2))
        self.register_buffer('buf', torch.zeros(2))
        self.plain = torch.zeros(2)

show("state_dict keys", lambda: list(M().state_dict().keys()))

In [ ]:
reveal(12)

---# Part 2 — Generation (5 items)**Blank editor. No documentation, no search.** Implement each function, then run its test.Each test compares your implementation against the real PyTorch builtin (or a naive loopreference) over 200 randomized shapes, including size-1 and size-0 dimensions. The oracle is`torch`, so a PASS is ground truth and not my opinion.If you cannot produce a function at all, that is the single most important signal in thisdiagnostic. Write down `blank` for it and move on rather than looking anything up.

In [ ]:
def diff_test(name, fn, ref, gen, trials=200, exact=True):
    torch.manual_seed(0)
    for t in range(trials):
        args = gen()
        try:
            expected = ref(*args)
        except Exception:
            continue
        got = fn(*args)
        if not isinstance(got, torch.Tensor):
            raise AssertionError(f"{name}: returned {type(got).__name__}, expected Tensor")
        if got.shape != expected.shape:
            raise AssertionError(f"{name}: shape {tuple(got.shape)} != {tuple(expected.shape)} (trial {t})")
        ok = torch.equal(got, expected) if exact else torch.allclose(got.float(), expected.float(), atol=1e-5)
        if not ok:
            raise AssertionError(f"{name}: values differ on trial {t}\n got={got}\n exp={expected}")
    print(f"{name}: PASS ({trials} randomized trials)")

def rand_shape(max_rank=3, max_dim=4, min_dim=0):
    r = torch.randint(1, max_rank + 1, (1,)).item()
    return tuple(torch.randint(min_dim, max_dim + 1, (r,)).tolist())

### G1. `one_hot(idx, C)``idx` is an integer tensor of **arbitrary shape**. Return a tensor of shape `(*idx.shape, C)`,dtype `torch.int64`, with a 1 at the index position.This is the question you were asked in a real interview. There are at least four routes.After it passes, name a second route, and say which of the two you would use at `C = 128000`and why.

In [ ]:
def one_hot(idx, C):
    raise NotImplementedError

def _gen_onehot():
    C = torch.randint(1, 8, (1,)).item()
    return (torch.randint(0, C, rand_shape()), C)

diff_test("G1 one_hot", one_hot, lambda i, C: F.one_hot(i, C), _gen_onehot)

### G2. `gather_last(x, idx)``x` has shape `(B, N)`, `idx` has shape `(B, K)` with entries in `[0, N)`.Return shape `(B, K)` where `out[b, k] = x[b, idx[b, k]]`.Do it with indexing, not with `torch.gather`. This is P5's rule used in anger.

In [ ]:
def gather_last(x, idx):
    raise NotImplementedError

def _gen_gather():
    b = torch.randint(1, 5, (1,)).item()
    n = torch.randint(1, 6, (1,)).item()
    k = torch.randint(1, 6, (1,)).item()
    return (torch.randn(b, n), torch.randint(0, n, (b, k)))

diff_test("G2 gather_last", gather_last, lambda x, i: torch.gather(x, -1, i), _gen_gather)

### G3. `causal_mask(s)`Return a bool tensor of shape `(s, s)` where entry `[i, j]` is True when query position `i`is allowed to attend to key position `j`.Build it from `arange` and a comparison. Do not use `tril`, `triu`, `ones`, or `eye`.Watch the boundary: is the diagonal allowed?

In [ ]:
def causal_mask(s):
    raise NotImplementedError

diff_test("G3 causal_mask", causal_mask,
          lambda s: torch.tril(torch.ones(s, s)).bool(),
          lambda: (torch.randint(1, 9, (1,)).item(),))

### G4. `expand_kv(k, n_rep)``k` has shape `(B, H_kv, S, D)`. Return shape `(B, H_kv * n_rep, S, D)` where each KV head isrepeated `n_rep` times, matching GQA's grouping convention: query head `i` attends to KV head`floor(i / n_rep)`.Get the grouping direction right. Your notes already flag this: the split is `(H_kv, G)`,not `(G, H_kv)`. The test will catch the wrong one.Bonus after it passes: is your version a view or a copy? Should it be?

In [ ]:
def expand_kv(k, n_rep):
    raise NotImplementedError

def _gen_gqa():
    b     = torch.randint(1, 3, (1,)).item()
    hkv   = torch.randint(1, 4, (1,)).item()
    n_rep = torch.randint(1, 4, (1,)).item()
    s     = torch.randint(1, 4, (1,)).item()
    d     = torch.randint(1, 4, (1,)).item()
    return (torch.randn(b, hkv, s, d), n_rep)

diff_test("G4 expand_kv", expand_kv,
          lambda k, n: k.repeat_interleave(n, dim=1), _gen_gqa)

### G5. `masked_mean(x, lengths)``x` has shape `(B, S, D)`, `lengths` has shape `(B,)` with each entry in `[1, S]`.Return shape `(B, D)`: the mean over the first `lengths[b]` positions of each row.No Python loop over the batch. The reference is a loop, so your job is to vectorise it.

In [ ]:
def masked_mean(x, lengths):
    raise NotImplementedError

def _masked_ref(x, lengths):
    out = torch.zeros(x.shape[0], x.shape[2])
    for i in range(x.shape[0]):
        out[i] = x[i, :lengths[i]].mean(0)
    return out

def _gen_masked():
    b = torch.randint(1, 5, (1,)).item()
    s = torch.randint(1, 7, (1,)).item()
    d = torch.randint(1, 4, (1,)).item()
    return (torch.randn(b, s, d), torch.randint(1, s + 1, (b,)))

diff_test("G5 masked_mean", masked_mean, _masked_ref, _gen_masked, exact=False)

---# Part 3 — Assertion writing (3 items)Each item gives you a `correct` and a `buggy` implementation of the same function. You do nothave to fix the bug. **Write the assertion you would have put in the code to catch it.**Your assertion must be silent on the correct implementation and must fire on the buggy one.The harness checks both directions, so an assertion that just says `assert True` fails, and sodoes one that is too strict.This is the bridge to F5 (debug-given-code) and it is the same discipline as your existingpre-submit ritual: assert the invariant, do not eyeball the tensor.

In [ ]:
def check_assertion(name, assert_fn, correct, buggy, gen, trials=50):
    torch.manual_seed(1)
    caught = 0
    for t in range(trials):
        args = gen()
        a1 = [a.clone() if torch.is_tensor(a) else a for a in args]
        try:
            assert_fn(correct(*a1), *a1)
        except AssertionError as e:
            raise AssertionError(f"{name}: your assertion fired on the CORRECT implementation "
                                 f"(trial {t}): {e}")
        a2 = [a.clone() if torch.is_tensor(a) else a for a in args]
        try:
            assert_fn(buggy(*a2), *a2)
        except AssertionError:
            caught += 1
    if caught == 0:
        raise AssertionError(f"{name}: your assertion never caught the bug in {trials} trials.")
    print(f"{name}: PASS (silent on correct, caught bug {caught}/{trials})")

### A1. Normalisation over the wrong axis`correct` normalises over the last axis. `buggy` normalises over the batch axis. On a squareinput these are the same shape and both look like probabilities.

In [ ]:
a1_correct = lambda x: F.softmax(x, dim=-1)
a1_buggy   = lambda x: F.softmax(x, dim=0)
a1_gen     = lambda: (torch.randn(torch.randint(2,5,(1,)).item(),
                                 torch.randint(2,6,(1,)).item()),)

def a1_assert(out, x):
    raise NotImplementedError

check_assertion("A1 softmax axis", a1_assert, a1_correct, a1_buggy, a1_gen)

### A2. Returns a view and mutates its argument`correct` returns a new tensor. `buggy` scales in place and returns its own input, so thecaller's tensor is silently modified and the two now alias.

In [ ]:
a2_correct = lambda x: x * 2
def a2_buggy(x):
    return x.mul_(2)
a2_gen = lambda: (torch.randn(torch.randint(1,5,(1,)).item(), 3),)

def a2_assert(out, x):
    raise NotImplementedError

check_assertion("A2 aliasing", a2_assert, a2_correct, a2_buggy, a2_gen)

### A3. Half-open interval on the diagonal`correct` allows a position to attend to itself. `buggy` uses a strict inequality. Both returna bool `(s, s)` tensor that looks like a causal mask.

In [ ]:
def a3_correct(s):
    i = torch.arange(s).unsqueeze(-1)
    return torch.arange(s) <= i

def a3_buggy(s):
    i = torch.arange(s).unsqueeze(-1)
    return torch.arange(s) < i

a3_gen = lambda: (torch.randint(1, 9, (1,)).item(),)

def a3_assert(out, s):
    raise NotImplementedError

check_assertion("A3 causal boundary", a3_assert, a3_correct, a3_buggy, a3_gen)

---# Scoring and miss profileFill in the three dicts and run. The output is what drives F5's bug-injection set, so record`blank` honestly for anything in Part 2 you could not start.

In [ ]:
part1 = {n: None for n in range(1, 13)}   # True / False
part2 = {n: None for n in ["G1","G2","G3","G4","G5"]}   # "pass" / "fail" / "blank"
part3 = {n: None for n in ["A1","A2","A3"]}             # "pass" / "fail" / "blank"

FAMILY = {1:"stride/view", 2:"stride/view", 3:"broadcasting", 4:"broadcasting/reduction",
          5:"indexing", 6:"aliasing", 7:"gather", 8:"scatter", 9:"dtype",
          10:"einsum/matmul", 11:"autograd", 12:"module state"}

def report():
    fams = {}
    for n, ok in part1.items():
        if ok is None: continue
        f = FAMILY[n]
        fams.setdefault(f, [0, 0])
        fams[f][1] += 1
        fams[f][0] += int(bool(ok))
    p1 = sum(1 for v in part1.values() if v is True)
    p1n = sum(1 for v in part1.values() if v is not None)
    p2 = sum(1 for v in part2.values() if v == "pass")
    p3 = sum(1 for v in part3.values() if v == "pass")
    blanks = [k for k, v in part2.items() if v == "blank"]
    print(f"Part 1 (recognition): {p1}/{p1n}")
    print(f"Part 2 (retrieval)  : {p2}/5   blank: {blanks or 'none'}")
    print(f"Part 3 (assertions) : {p3}/3")
    print()
    print("Miss profile by family:")
    for f, (c, t) in sorted(fams.items(), key=lambda kv: kv[1][0] / max(kv[1][1], 1)):
        flag = "  <-- target" if c < t else ""
        print(f"  {f:<24} {c}/{t}{flag}")
    print()
    if p1n and p1 / p1n >= 0.75 and p2 <= 2:
        print("READ: recognition strong, retrieval weak. This is the predicted profile.")
        print("      Weight F2 and F3 heavily; Tensor Puzzles will feel easy and teach little.")
    elif p1n and p1 / p1n < 0.6:
        print("READ: recognition gaps too. Do F1 (Tensor Puzzles) before F2.")

report()